In [27]:
!pip install pulp

  Using cached pulp-3.3.0-py3-none-any.whl.metadata (8.4 kB)
Using cached pulp-3.3.0-py3-none-any.whl (16.4 MB)


In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt

In [2]:
prices = pd.read_csv('/Users/marysia/Desktop/applied/OR/food_prices_200.xlsx - food_prices.csv')
nutrients = pd.read_csv('/Users/marysia/Desktop/applied/OR/food_nutrition_200.xlsx - food_nutrition_per_100g.csv')

In [6]:
prices.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 205 entries, 0 to 204
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   food_id                 205 non-null    int64  
 1   food_name               205 non-null    object 
 2   category                205 non-null    object 
 3   subcategory             205 non-null    object 
 4   market_unit             205 non-null    object 
 5   price_per_unit_usd      205 non-null    float64
 6   unit_weight_g           205 non-null    object 
 7   price_per_100g_usd      205 non-null    float64
 8   price_per_1g_usd        205 non-null    float64
 9   bls_series_or_source    205 non-null    object 
 10  price_reference_period  204 non-null    object 
 11  official_data_url       205 non-null    object 
 12  notes                   205 non-null    object 
dtypes: float64(3), int64(1), object(9)
memory usage: 20.9+ KB


In [5]:
nutrients.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 205 entries, 0 to 204
Data columns (total 35 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   food_id                205 non-null    int64  
 1   food_name              205 non-null    object 
 2   category               205 non-null    object 
 3   subcategory            205 non-null    object 
 4   fdc_id                 205 non-null    int64  
 5   preparation_state      205 non-null    object 
 6   energy_kcal            205 non-null    int64  
 7   protein_g              205 non-null    float64
 8   total_fat_g            205 non-null    float64
 9   carbohydrates_g        205 non-null    float64
 10  dietary_fiber_g        205 non-null    float64
 11  sugars_g               205 non-null    float64
 12  saturated_fat_g        205 non-null    float64
 13  monounsaturated_fat_g  205 non-null    float64
 14  polyunsaturated_fat_g  205 non-null    float64
 15  choles

In [23]:
required = pd.read_csv('/Users/marysia/Desktop/applied/OR/last_version_prpcessed.csv')

In [24]:
required.head()

,Nutrient,7–9 years,10–12 years,13–15 years,lower_7-9,lower_10-12,lower_13-15,upper_7-9,upper_10-12,upper_13-15
0,Energy [kcal/day],1800.0,2200.0,2700.0,NaN,NaN,NaN,NaN,NaN,NaN
1,Protein [g/day],30.0,42.0,58.0,NaN,NaN,NaN,NaN,NaN,NaN
2,Sodium [mg/day],1200.0,1300.0,1500.0,NaN,NaN,NaN,NaN,NaN,NaN
3,Carbohydrates [% energy],50.0,50.0,50.0,45.0,45.0,45.0,65.0,65.0,65.0
4,Fiber [g/day],16.0,19.0,19.0,NaN,NaN,NaN,NaN,NaN,NaN


In [30]:
import pandas as pd
from pulp import *

# 1. WCZYTANIE DANYCH
df_norms = pd.read_csv('last_version_prpcessed.csv')
df_nutrition = pd.read_csv('food_nutrition_200.xlsx - food_nutrition_per_100g.csv')
df_prices = pd.read_csv('food_prices_200.xlsx - food_prices.csv')

# 2. OCZYSZCZANIE DANYCH (Kluczowe dla uniknięcia TypeError)
# Lista kolumn z wartościami odżywczymi do konwersji
nutrient_cols = ['energy_kcal', 'protein_g', 'total_fat_g', 'carbohydrates_g', 'sodium_mg', 'dietary_fiber_g']

# Konwersja kolumn w produktach na liczby, błędy zamieniamy na 0
for col in nutrient_cols:
    df_nutrition[col] = pd.to_numeric(df_nutrition[col], errors='coerce').fillna(0)

# Konwersja ceny na liczby
df_prices['price_per_100g_usd'] = pd.to_numeric(df_prices['price_per_100g_usd'], errors='coerce').fillna(0)

# Połączenie cen z wartościami odżywczymi
df_products = pd.merge(df_nutrition, df_prices[['food_id', 'price_per_100g_usd']], on='food_id')

def solve_diet_simplex(age_group_label):
    # Oczyszczenie nazwy grupy i normalizacja myślnika
    clean_label = age_group_label.strip()
    suffix = clean_label.split(' ')[0].replace('–', '-') 
    
    # Konwersja kolumn w normach na liczby (dla wybranej grupy)
    cols_to_fix = [age_group_label, f'lower_{suffix}', f'upper_{suffix}']
    for c in cols_to_fix:
        if c in df_norms.columns:
            df_norms[c] = pd.to_numeric(df_norms[c], errors='coerce')

    # Pobranie zapotrzebowania kalorycznego
    energy_row = df_norms[df_norms['Nutrient'].str.contains('Energy', na=False)]
    total_kcal = float(energy_row[age_group_label].values[0])

    # Inicjalizacja modelu
    prob = LpProblem(f"Dieta_{suffix}", LpMinimize)

    # Zmienne: ilość gramów produktu (limit 500g na produkt)
    food_items = df_products['food_name'].tolist()
    food_vars = LpVariable.dicts("gram", food_items, lowBound=0, upBound=500, cat='Continuous')

    # Funkcja celu: koszt
    prob += lpSum([(food_vars[row['food_name']] / 100) * row['price_per_100g_usd'] for _, row in df_products.iterrows()])

    # Mapowanie składników
    nutrient_map = {
        'Energy [kcal/day]': ('energy_kcal', 1),
        'Protein [g/day]': ('protein_g', 1),
        'Fat [% energy]': ('total_fat_g', 9),
        'Carbohydrates [% energy]': ('carbohydrates_g', 4),
        'Sodium [mg/day]': ('sodium_mg', 1),
        'Fiber [g/day]': ('dietary_fiber_g', 1)
    }

    print(f"\n--- Grupa: {clean_label} ({total_kcal} kcal) ---")

    for norm_name, (food_col, kcal_factor) in nutrient_map.items():
        if norm_name not in df_norms['Nutrient'].values:
            continue
            
        row = df_norms[df_norms['Nutrient'] == norm_name].iloc[0]
        prop_val = float(row[age_group_label])
        
        # Pobieranie limitów
        lb_val = row.get(f'lower_{suffix}')
        ub_val = row.get(f'upper_{suffix}')
        
        # Twoja logika limitów (0.3 / 1.0)
        final_lb = float(lb_val) if pd.notnull(lb_val) else prop_val * 0.3
        final_ub = float(ub_val) if pd.notnull(ub_val) else prop_val * 1.0

        # Przeliczenie % na gramy
        if '% energy' in norm_name:
            final_lb = (final_lb * total_kcal / 100) / kcal_factor
            final_ub = (final_ub * total_kcal / 100) / kcal_factor

        # Dodanie ograniczenia: suma składnika w diecie musi być w przedziale [LB, UB]
        total_in_diet = lpSum([(food_vars[r['food_name']] / 100) * float(r[food_col]) for _, r in df_products.iterrows()])
        
        prob += total_in_diet >= final_lb, f"Min_{food_col}"
        prob += total_in_diet <= final_ub, f"Max_{food_col}"

    # Rozwiązanie
    status = prob.solve(PULP_CBC_CMD(msg=0))

    if LpStatus[status] == 'Optimal':
        results = []
        for item in food_items:
            val = food_vars[item].varValue
            if val and val > 0.1:
                results.append({"Produkt": item, "Gramy": round(val, 2)})
        
        print(pd.DataFrame(results).to_string(index=False))
        print(f"Koszt: ${round(value(prob.objective), 2)}")
    else:
        print("BŁĄD: Nie znaleziono diety spełniającej limity.")

# Uruchomienie
groups = ['7–9 years ', '10–12 years ', '13–15 years']
for g in groups:
    solve_diet_simplex(g)


--- Grupa: 7–9 years (1800.0 kcal) ---
                            Produkt  Gramy
    Pasta, Spaghetti, Dry, Enriched  40.90
                    Watermelon, Raw 500.00
                Mayonnaise, Regular  42.79
                         Water, Tap 500.00
Cornmeal, Whole Grain, Yellow (dry) 173.85
Koszt: $0.93

--- Grupa: 10–12 years (2200.0 kcal) ---
                             Produkt  Gramy
     Pasta, Spaghetti, Dry, Enriched  73.54
                     Watermelon, Raw 500.00
Vegetable Oil (Soybean/Canola blend)   5.27
                 Mayonnaise, Regular  45.73
                          Water, Tap 500.00
 Cornmeal, Whole Grain, Yellow (dry) 200.64
Koszt: $1.15

--- Grupa: 13–15 years (2700.0 kcal) ---
                             Produkt  Gramy
     Pasta, Spaghetti, Dry, Enriched 210.60
                     Watermelon, Raw 500.00
Vegetable Oil (Soybean/Canola blend)   7.89
                 Mayonnaise, Regular  57.19
                          Water, Tap 500.00
 Cornmeal, Whole Gra